# Ayudantía 11 SQL, extracción de datos y dashboards

In [60]:
import json
import sqlite3
import time
import urllib.request

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup

## Parte 1. SQL con una base de películas

La celda siguiente ya viene resuelta. Repite lo visto en el capítulo 1. Descarga `movies.json`, crea las cinco tablas con sus llaves y carga los datos usando diccionarios para no duplicar géneros ni actores. Basta con ejecutarla y revisar los conteos finales.

Las tablas son `Movies`, `Actors`, `Genres` y las dos tablas puente `ActorsMovies` y `GenresMovies`.

In [61]:
URL = ('https://raw.githubusercontent.com/IIC2115/Syllabus/main/'
       'Material%20de%20clases/Cap%C3%ADtulo%204/Notebooks/movies.json')
movies = json.loads(urllib.request.urlopen(URL).read().decode('utf-8'))

# Iniciamos con la conexión a SQLite
connection = sqlite3.connect(':memory:')
cursor = connection.cursor()

# Restricciones
cursor.execute("PRAGMA foreign_keys = ON")

# Creación Tablas
cursor.execute("CREATE TABLE Movies(mid INTEGER PRIMARY KEY, title TEXT NOT NULL, year INTEGER)")
cursor.execute("CREATE TABLE Actors(aid INTEGER PRIMARY KEY, name TEXT NOT NULL)")
cursor.execute("CREATE TABLE Genres(gid INTEGER PRIMARY KEY, genre TEXT NOT NULL)")
cursor.execute("""CREATE TABLE ActorsMovies(
    actor_id INTEGER,
    movie_id INTEGER,
    FOREIGN KEY (actor_id) REFERENCES Actors(aid),
    FOREIGN KEY (movie_id) REFERENCES Movies(mid)
)""")
cursor.execute("""CREATE TABLE GenresMovies(
    genre_id INTEGER,
    movie_id INTEGER,
    FOREIGN KEY (genre_id) REFERENCES Genres(gid),
    FOREIGN KEY (movie_id) REFERENCES Movies(mid)
)""")

# Poblado inicial
mid, aid, gid = 1, 1, 1
genres_dict, actors_dict = {}, {}
for movie in movies:
    cursor.execute("INSERT INTO Movies(title, year) VALUES (?, ?)", (movie['title'], movie['year']))
    for genre in movie['genres']:
        if genre not in genres_dict:
            genres_dict[genre] = gid
            cursor.execute("INSERT INTO Genres(genre) VALUES (?)", (genre,))
            gid += 1
        cursor.execute("INSERT INTO GenresMovies VALUES (?, ?)", (genres_dict[genre], mid))
    for actor in movie['cast']:
        if actor not in actors_dict:
            actors_dict[actor] = aid
            cursor.execute("INSERT INTO Actors(name) VALUES (?)", (actor,))
            aid += 1
        cursor.execute("INSERT INTO ActorsMovies VALUES (?, ?)", (actors_dict[actor], mid))
    mid += 1
connection.commit()

for tabla in ['Movies', 'Actors', 'Genres', 'ActorsMovies', 'GenresMovies']:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    print(tabla, cursor.fetchone()[0])

Movies 28795
Actors 15531
Genres 41
ActorsMovies 82881
GenresMovies 33009


### Ejercicio 1.1. Películas de un año

Cuenta cuántas películas son del año 2000. Se necesita `SELECT`, `FROM` y `WHERE`.

**Pista.** El año está en la columna `year` de la tabla `Movies`.

In [62]:
cursor.execute("SELECT COUNT(*) FROM Movies WHERE year = 2000")
print(cursor.fetchone()[0])   # debería mostrar 213

213


### Ejercicio 1.2. Comedias de 1995

Para llegar de las películas a los géneros hay que pasar por la tabla puente, así que se necesitan dos `JOIN`. Cuenta las comedias de 1995.

**Pista.** Cada `JOIN` avanza por una llave foránea, de `Movies` a `GenresMovies` y de ahí a `Genres`.

In [63]:
cursor.execute("""
    SELECT M.title
    FROM Movies M
    JOIN GenresMovies GM ON M.mid = GM.movie_id
    JOIN Genres G ON GM.genre_id = G.gid
    WHERE G.genre = 'Comedy' AND M.year = 1995
""")
resultados = cursor.fetchall()
print(len(resultados))   # debería mostrar 73

73


### Ejercicio 1.3. Los géneros más grandes

Cuenta las películas de cada género y deja solo los géneros con más de 5000 películas, ordenados de mayor a menor.

**Pista.** `WHERE` filtra filas antes de agrupar y `HAVING` filtra los grupos después de agrupar.

In [64]:
cursor.execute("""
    SELECT G.genre, COUNT(*) AS cantidad
    FROM Genres G
    JOIN GenresMovies GM ON G.gid = GM.genre_id
    GROUP BY G.genre
    HAVING COUNT(*) > 5000
    ORDER BY cantidad DESC
""")
print(cursor.fetchall())

[('Drama', 8744), ('Comedy', 7363)]


### Ejercicio 1.4. Consultas con parámetros

Escribe una función que cuente las películas de cualquier año. El valor debe entrar a la consulta con `?` y no pegado en el texto.

**Pista.** El segundo argumento de `execute` es una tupla, y una tupla de un solo valor lleva una coma al final.

In [65]:
def peliculas_del_anio(anio):
    cursor.execute("SELECT COUNT(*) FROM Movies WHERE year = ?", (anio,))
    return cursor.fetchone()[0]

print(peliculas_del_anio(2000))
print(peliculas_del_anio(1995))

213
307


### Ejercicio 1.5. La llave foránea se defiende

Intenta insertar una relación con un actor que no existe y captura el error. Recuerda que la base tiene activado `PRAGMA foreign_keys = ON`.

**Pista.** El error se llama `sqlite3.IntegrityError`.

In [66]:
try:
    cursor.execute("INSERT INTO ActorsMovies VALUES (999999, 1)")
    print("se insertó sin problemas")
except sqlite3.IntegrityError as e:
    print("error de integridad.", e)

error de integridad. FOREIGN KEY constraint failed


### Desafío 1.6. Los cinco actores con más películas

Une `Actors` con `ActorsMovies`, agrupa por actor, cuenta y muestra solo los cinco primeros.

**Pista.** Se combinan `GROUP BY`, `ORDER BY ... DESC` y `LIMIT`.

In [67]:
cursor.execute("""
    SELECT A.name, COUNT(*) AS peliculas
    FROM Actors A
    JOIN ActorsMovies AM ON A.aid = AM.actor_id
    GROUP BY A.name
    ORDER BY peliculas DESC
    LIMIT 5
""")
for fila in cursor.fetchall():
    print(fila)

('Harold Lloyd', 190)
('Hoot Gibson', 142)
('John Wayne', 136)
('Charles Starrett', 116)
('Bebe Daniels', 103)


## Parte 2. Extracción de datos

Aquí los datos no vienen en un archivo. Primero se lee una tabla de Wikipedia con `pd.read_html`, luego se recorre el HTML de una página con BeautifulSoup y al final se piden datos a una API.

### Ejercicio 2.1. Una tabla de Wikipedia

Lee todas las tablas de la página de países de Sudamérica por población y quédate con la que tiene más filas.

**Pista.** `pd.read_html` devuelve una lista de DataFrames, y `shape[0]` es el número de filas de cada uno. Se agrega un `User-Agent` para evitar el error 403.

In [68]:
url = 'https://en.wikipedia.org/wiki/List_of_South_American_countries_by_population'
tablas = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})
print(f"tablas encontradas: {len(tablas)}")

tabla_principal = max(tablas, key=lambda t: t.shape[0])
print(tabla_principal.shape)
tabla_principal.head()

tablas encontradas: 8
(15, 7)


,Unnamed: 0,Country / dependency,% total,South America population[1],% change,Official figure,Official date
0,1.0,Brazil,49.2%,216422446,0.5%,216284269,1 Jul 2023[2]
1,2.0,Colombia,11.8%,52085168,0.4%,52215503,17 Mar 2023[3]
2,3.0,Argentina,10.4%,45773884,0.6%,46044703,18 May 2022[4]
3,4.0,Peru,7.8%,34352719,0.9%,33725844,30 Jun 2023[5]
4,5.0,Venezuela,6.6%,28838499,1.9%,32219500,30 Jun 2019[6]


### Ejercicio 2.2. El HTML de una página

Descarga la página de Chile en Wikipedia, revisa el código de respuesta y extrae el título, los encabezados `h2` y los primeros links internos.

**Pista.** Los links internos de Wikipedia empiezan con `/wiki/`.

In [69]:
url = 'https://en.wikipedia.org/wiki/Chile'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
print(response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.text)

encabezados = soup.find_all('h2')
print(len(encabezados), "encabezados h2")

links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].startswith('/wiki/')]
print(links[:5])

200
Chile - Wikipedia
12 encabezados h2
['/wiki/Main_Page', '/wiki/Wikipedia:Contents', '/wiki/Portal:Current_events', '/wiki/Special:Random', '/wiki/Wikipedia:About']


### Ejercicio 2.3. Una solicitud a una API

Pide a Open-Meteo el clima actual de Santiago. La API no necesita API key ni registro.

**Pista.** La longitud de Santiago es -70.67, y los datos pedidos vienen dentro del bloque `current` de la respuesta.

In [70]:
url = 'https://api.open-meteo.com/v1/forecast'
params = {'latitude': -33.45, 'longitude': -70.67, 'current': 'temperature_2m,wind_speed_10m'}
response = requests.get(url, params=params)
print(response.status_code)

data = response.json()
print(data['current']['temperature_2m'])

200
23.3


### Ejercicio 2.4. Varias ciudades

Repite la solicitud para tres ciudades, guarda solo las respuestas exitosas, deja una pausa entre solicitudes y ordena el resultado de la más cálida a la más fría.

**Pista.** El código 200 significa éxito. La pausa se hace con `time.sleep`.

In [71]:
ciudades = {'Santiago': (-33.45, -70.67), 'Lima': (-12.05, -77.04), 'Bogotá': (4.71, -74.07)}
resultados = []
for ciudad, (lat, lon) in ciudades.items():
    params = {'latitude': lat, 'longitude': lon, 'current': 'temperature_2m'}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        temperatura = response.json()['current']['temperature_2m']
        resultados.append({'ciudad': ciudad, 'temperatura': temperatura})
    time.sleep(0.5)

df_clima = pd.DataFrame(resultados).sort_values('temperatura', ascending=False)
df_clima

,ciudad,temperatura
1,Lima,26.6
0,Santiago,23.3
2,Bogotá,21.8


## Parte 3. El dashboard de clima con Dash

En esta parte se arma el dashboard que muestran las láminas del capítulo 3, con sus dos pestañas. La pestaña Home presenta los datos y el Explorador climático permite elegir país, ciudad, fechas y variables para ver métricas, información geográfica y un gráfico.

Primero se prepara el dato, después se practican los componentes y Plotly Express, y al final se escribe el layout y los dos callbacks de la app. Como los datos ya se trabajaron con Pandas, la preparación viene resuelta.

In [72]:
!pip install dash dash-bootstrap-components

In [73]:
import os
import zipfile

URL_CLIMA = ('https://raw.githubusercontent.com/IIC2115/Syllabus/main/'
             'Material%20de%20clases/Cap%C3%ADtulo%204/Notebooks/weather_data.zip')
if not os.path.exists('weather_data.db'):
    urllib.request.urlretrieve(URL_CLIMA, 'weather_data.zip')
    with zipfile.ZipFile('weather_data.zip') as z:
        z.extractall('.')

conn = sqlite3.connect('weather_data.db')
df_weather = pd.read_sql_query("SELECT * FROM weather", conn)
df_countries = pd.read_sql_query("SELECT * FROM countries", conn)
df_cities = pd.read_sql_query("SELECT * FROM cities", conn)
conn.close()

df_weather = df_weather.dropna()

# Se unen las mediciones con las ciudades y luego con los países
weather_cities = df_weather.merge(df_cities.drop(columns=["city_name"]), on='station_id')
df_countries_renamed = df_countries.rename(columns={"country": "country_name"})
full_df = weather_cities.merge(
    df_countries_renamed.drop(columns=["iso3", "capital", "capital_lat", "capital_lng", "native_name"]),
    on='iso2'
)
df_weather_fullinfo = full_df.dropna(subset=["avg_temp_c", "city_name", "country_name", "date"]).copy()
df_weather_fullinfo['year'] = pd.to_datetime(df_weather_fullinfo['date']).dt.year

# Datos que usa la app
df = df_weather_fullinfo.copy()
variables = ["avg_temp_c", "precipitation_mm", "avg_wind_speed_kmh", "sunshine_total_min"]
paises = df['country'].unique()
print(len(df), "registros,", df['city_name'].nunique(), "ciudades,", df['country'].nunique(), "países")

397551 registros, 57 ciudades, 8 países


In [74]:
# Ícono de la app, para no depender de un archivo externo
import base64
from PIL import Image, ImageDraw

icono = Image.new('RGBA', (160, 160), (0, 0, 0, 0))
dibujo = ImageDraw.Draw(icono)
dibujo.ellipse((50, 30, 110, 90), fill=(250, 190, 40, 255))
for caja in [(20, 70, 90, 130), (60, 60, 140, 130), (40, 85, 120, 140)]:
    dibujo.ellipse(caja, fill=(230, 235, 240, 255))
icono.save('icon.png')
encoded_image = base64.b64encode(open('icon.png', 'rb').read()).decode('ascii')

In [86]:
from dash import Dash, dcc, html, dash_table, Input, Output, State
import dash_bootstrap_components as dbc

def ejecutar(app, puerto, alto=650):
    app.run(port=puerto, debug=False, jupyter_mode='inline', jupyter_height=alto)

### Ejercicio 3.1. Componentes para elegir y accionar

Completa los nombres de los componentes que muestran las láminas. Los tres primeros eligen opciones, los dos siguientes eligen números, el selector de fechas elige un intervalo y el botón ejecuta una acción.

**Pista.** Los nombres son `Dropdown`, `RadioItems`, `Checklist`, `Slider`, `RangeSlider` y `DatePickerRange`. El botón es de `html` y se llama `Button`.

In [76]:
app_demo = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app_demo.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader(html.H5("Formulario de Prueba de Componentes", className="text-center text-primary m-0")),
                dbc.CardBody([
                    html.Div([
                        html.Label("Ciudad", style={"fontWeight": "bold"}),
                        dcc.Dropdown(options=['Santiago', 'Lima', 'Bogotá'], value='Santiago', id='ciudad', className="mb-3")
                    ]),

                    html.Div([
                        html.Label("Frecuencia", style={"fontWeight": "bold"}),
                        dcc.RadioItems(
                            options=['Diaria', 'Mensual'],
                            value='Diaria',
                            id='frecuencia',
                            labelStyle={'display': 'inline-block', 'marginRight': '15px'},
                            className="mb-3"
                        )
                    ]),

                    html.Div([
                        html.Label("Variables", style={"fontWeight": "bold"}),
                        dcc.Checklist(
                            options=['Temperatura', 'Lluvia', 'Viento'],
                            value=['Temperatura'],
                            id='vars',
                            labelStyle={'display': 'inline-block', 'marginRight': '15px'},
                            className="mb-3"
                        )
                    ]),

                    html.Div([
                        html.Label("Temperatura Máxima", style={"fontWeight": "bold"}),
                        dcc.Slider(0, 30, 5, value=10, id='slider', className="mb-3")
                    ]),

                    html.Div([
                        html.Label("Rango de Temperatura", style={"fontWeight": "bold"}),
                        dcc.RangeSlider(0, 30, 5, value=[5, 20], id='rango', className="mb-3")
                    ]),

                    html.Div([
                        html.Label("Fechas", style={"fontWeight": "bold"}),
                        html.Br(),
                        dcc.DatePickerRange(start_date='2020-01-01', end_date='2020-12-31', id='fechas', className="mb-4")
                    ]),

                    dbc.Button("Actualizar", id='boton', color="primary", className="w-100", n_clicks=0)
                ])
            ], style={"boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "borderRadius": "10px"})
        ], width=12, md=8, lg=6, className="mx-auto mt-4")
    ])
], fluid=True)

ejecutar(app_demo, 8051, 800)

<IPython.core.display.Javascript object>

### Ejercicio 3.2. Mostrar resultados

Muestra un gráfico de Plotly y una tabla con la temperatura de Denver durante 1990.

**Pista.** El gráfico se muestra con `dcc.Graph` y la tabla con `dash_table.DataTable`.

In [77]:
y90 = df[(df['city_name'] == 'Denver') & (df['year'] == 1990)].sort_values('date')
fig = px.line(y90, x='date', y='avg_temp_c', title='Temperatura diaria en Denver, 1990')

app_res = Dash(__name__)
app_res.layout = html.Div([
    dcc.Graph(figure=fig, id='grafico'),
    dash_table.DataTable(y90[['date', 'avg_temp_c', 'precipitation_mm']].head(4).to_dict('records'), page_size=4),
])

ejecutar(app_res, 8052, 700)

<IPython.core.display.Javascript object>

### Ejercicio 3.3. Un dato y cuatro gráficos

Con los mismos datos de Denver, cambia solo la función de Plotly Express para obtener un gráfico de líneas, uno de barras por mes, uno de dispersión y un histograma.

**Pista.** Las funciones se llaman `line`, `bar`, `scatter` y `histogram`.

In [78]:
mensual = y90.groupby(pd.to_datetime(y90['date']).dt.month)['avg_temp_c'].mean().reset_index()
mensual.columns = ['mes', 'avg_temp_c']

px.line(y90, x='date', y='avg_temp_c', title='Líneas').show()
px.bar(mensual, x='mes', y='avg_temp_c', title='Barras por mes').show()
px.scatter(y90, x='avg_temp_c', y='precipitation_mm', title='Dispersión').show()
px.histogram(y90, x='avg_temp_c', title='Histograma').show()

### La app completa

Desde aquí se escribe la app que muestran las láminas 33 y 34. Se crea con el tema de Bootstrap, y el encabezado con el ícono viene resuelto.

In [79]:
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Clima Global"

encabezado = html.Div([
    html.Img(src=f'data:image/png;base64,{encoded_image}', height="60px", style={"marginRight": "15px"}),
    html.H1("Dashboard Clima Global", style={"margin": 0})
], style={"display": "flex", "alignItems": "center", "justifyContent": "center", "marginBottom": "20px"})

### Ejercicio 3.4. La pestaña Home

La pestaña Home es una tarjeta de bienvenida con estadísticas del conjunto de datos. Completa el conteo de ciudades y de países.

**Pista.** `nunique()` cuenta los valores distintos de una columna.

In [80]:
home = dbc.Container([
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.Div([
                        html.Img(src=f'data:image/png;base64,{encoded_image}', height="80px", style={"marginRight": "15px"}),
                        html.H3("Bienvenido al Dashboard de Clima Global", className="card-title", style={"margin": 0})
                    ], style={"display": "flex", "alignItems": "center", "marginBottom": "20px"}),
                    html.P("Este dashboard permite explorar datos climáticos diarios por ciudad, permitiendo comparar distintas variables ambientales históricas.", className="card-text"),
                    html.Hr(),
                    html.P(f"Número total de registros: {len(df):,}", className="card-text"),
                    html.P(f"Número de ciudades: {df['city_name'].nunique()}", className="card-text"),
                    html.P(f"Número de países: {df['country'].nunique()}", className="card-text"),
                    html.Hr(),
                    html.P("Fuente de datos: dataset procesado desde registros históricos de estaciones meteorológicas internacionales.")
                ])
            ], className="mt-4")
        ], width=8)
    ], justify="center")
])

### Ejercicio 3.5. Los controles del Explorador

Son los componentes numerados del 1 al 5 en la lámina 34, es decir los selectores de país y de ciudad, el rango de fechas, las variables y el botón. Completa el identificador del primer menú y los nombres de tres componentes.

**Pista.** El identificador del primer menú es `pais-dropdown`. Los componentes son `DatePickerRange`, `Checklist` y `Button`.

In [81]:
controles = dbc.Col([
    dbc.Card([
        dbc.CardHeader(html.H4("Filtros de Búsqueda", className="text-center text-primary"), style={"fontWeight": "bold"}),
        dbc.CardBody([
            # Selector de País
            html.Div([
                html.Label("Selecciona un País:", style={"fontWeight": "bold", "marginBottom": "5px"}),
                dcc.Dropdown(
                    id='pais-dropdown',
                    options=[{'label': pais, 'value': pais} for pais in sorted(paises)],
                    value="United States of America",
                    clearable=False,
                    className="mb-3"
                )
            ]),

            # Selector de Ciudad (se actualiza dinámicamente con el primer callback)
            html.Div([
                html.Label("Selecciona una Ciudad:", style={"fontWeight": "bold", "marginBottom": "5px"}),
                dcc.Dropdown(
                    id='ciudad-dropdown',
                    options=[],
                    value="Denver",
                    clearable=False,
                    className="mb-3"
                )
            ]),

            # Selector de Rango de Fechas
            html.Div([
                html.Label("Rango de Fechas:", style={"fontWeight": "bold", "marginBottom": "5px"}),
                html.Br(),
                dcc.DatePickerRange(
                    id='date-range',
                    start_date=df['date'].min(),
                    end_date=df['date'].max(),
                    display_format='YYYY-MM-DD',
                    className="mb-3"
                )
            ]),

            # Selector de Variables Climáticas (Checklist)
            html.Div([
                html.Label("Variables a Visualizar:", style={"fontWeight": "bold", "marginBottom": "5px"}),
                dcc.Checklist(
                    id='variable-checklist',
                    options=[{'label': f" {v.replace('_', ' ').title()}", 'value': v} for v in variables],
                    value=["avg_temp_c"],
                    labelStyle={'display': 'block', 'marginBottom': '5px'},
                    className="mb-3"
                )
            ]),

            # Botón de Acción estilizado con Bootstrap
            html.Div([
                dbc.Button(
                    "Actualizar Dashboard",
                    id="submit-button",
                    color="primary",
                    className="w-100 mt-2",
                    n_clicks=0
                )
            ])
        ])
    ], style={"boxShadow": "0 4px 8px 0 rgba(0,0,0,0.1)", "borderRadius": "10px"})
], width=12, lg=4, className="mb-4")

### Ejercicio 3.6: Maquetación y Layout Completo del Dashboard

Definimos la estructura principal de la interfaz utilizando pestañas para separar el inicio del explorador dinámico de datos de manera organizada.

In [82]:
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(encabezado, width=12, className="text-center mt-4 mb-4")
    ]),

    dcc.Tabs([
        dcc.Tab(
            label='Inicio (Home)',
            style={'padding': '12px', 'fontWeight': 'bold'},
            selected_style={'padding': '12px', 'fontWeight': 'bold', 'borderTop': '3px solid #0275d8'},
            children=[home]
        ),

        dcc.Tab(
            label='Explorador Climático',
            style={'padding': '12px', 'fontWeight': 'bold'},
            selected_style={'padding': '12px', 'fontWeight': 'bold', 'borderTop': '3px solid #0275d8'},
            children=[
                dbc.Container([
                    dbc.Row([
                        controles,
                        dbc.Col([
                            html.Div(id="info-ubicacion")
                        ], width=12, lg=8, className="mb-4")
                    ], className="mt-4", align="start"),

                    dbc.Row([
                        dbc.Col(html.Div(id="metricas"), width=12)
                    ], className="mb-4"),

                    dbc.Row([
                        dbc.Col([
                            dbc.Card([
                                dbc.CardHeader(html.H5("Visualización de Tendencias Históricas", className="text-primary m-0")),
                                dbc.CardBody(dcc.Graph(id="grafico"))
                            ], style={"boxShadow": "0 4px 8px 0 rgba(0,0,0,0.1)", "borderRadius": "10px"})
                        ], width=12)
                    ], className="mb-5")
                ], fluid=True)
            ]
        )
    ])
], fluid=True, style={"backgroundColor": "#f8f9fa", "minHeight": "100vh"})

### Ejercicio 3.7. Primer callback, las ciudades según el país

Cuando el usuario elige un país, el menú de ciudades debe mostrar solo las ciudades de ese país. El país entra como `Input` y las opciones de ciudad salen como `Output`.

**Pista.** Los identificadores son `ciudad-dropdown` y `pais-dropdown`, y el valor elegido está en la propiedad `value`.

In [83]:
@app.callback(
    Output('ciudad-dropdown', 'options'),
    Input('pais-dropdown', 'value')
)
def set_ciudades(pais):
    ciudades = df[df["country"] == pais]["city_name"].unique()
    return [{'label': ciudad, 'value': ciudad} for ciudad in sorted(ciudades)]

### Ejercicio 3.8: Callback de Actualización del Explorador Climático


In [84]:
@app.callback(
    [
        Output("grafico", "figure"),
        Output("metricas", "children"),
        Output("info-ubicacion", "children")
    ],
    Input("submit-button", "n_clicks"),
    [
        State("ciudad-dropdown", "value"),
        State("pais-dropdown", "value"),
        State("date-range", "start_date"),
        State("date-range", "end_date"),
        State("variable-checklist", "value")
    ]
)
def actualizar_vista(n_clicks, ciudad, pais, start_date, end_date, vars_sel):
    # Validacion: Si no hay variables o ciudad seleccionada, retornamos contenedores vacios con un mensaje preventivo
    if not vars_sel or not ciudad:
        return (
            {},
            dbc.Alert("Atención: Selecciona una ciudad y al menos una variable climática para graficar.", color="warning"),
            ""
        )

    # Filtrado del DataFrame original utilizando Pandas de forma optimizada
    dff = df[
        (df["city_name"] == ciudad) &
        (df["country"] == pais) &
        (df["date"] >= start_date) &
        (df["date"] <= end_date)
    ]

    # Si el cruce de filtros no produce datos en ese rango, mostramos una alerta al usuario
    if dff.empty:
        return (
            {},
            dbc.Alert("No se encontraron datos para los criterios seleccionados en este rango de fechas.", color="danger"),
            ""
        )

    # Creación del gráfico dinámico con Plotly Express
    fig = px.line(
        dff.sort_values("date"),
        x="date",
        y=vars_sel,
        title=f"Historial de Variables en {ciudad} ({pais})",
        labels={"value": "Valor de Medida", "date": "Fecha", "variable": "Variable"},
        template="plotly_white"
    )

    fig.update_layout(
        margin=dict(l=40, r=40, t=50, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    # Creación dinamica de Tarjetas de Métricas Estadísticas mediante Bootstrap
    tarjetas_list = []
    for var in vars_sel:
        promedio = dff[var].mean()
        maximo = dff[var].max()
        nombre_formateado = var.replace("_", " ").title()

        tarjeta = dbc.Col([
            dbc.Card([
                dbc.CardHeader(html.H5(nombre_formateado, className="text-center text-white m-0"), className="bg-primary"),
                dbc.CardBody([
                    html.H6("Resumen del Período:", className="card-subtitle mb-2 text-muted"),
                    html.P([html.Strong("Promedio: "), f"{promedio:.2f}"], className="card-text"),
                    html.P([html.Strong("Valor Máximo: "), f"{maximo:.2f}"], className="card-text")
                ])
            ], style={"boxShadow": "0 4px 6px rgba(0,0,0,0.05)", "borderRadius": "8px"})
        ], width=12, md=6, lg=3, className="mb-3")

        tarjetas_list.append(tarjeta)

    # Encapsulamos las columnas en una fila responsiva
    contenedor_metricas = dbc.Row(tarjetas_list)

    # Obtención de la información geografica de la primera fila coincidente
    ubicacion = dff.iloc[0][["region", "continent", "latitude", "longitude"]]

    # Diseño de la tarjeta de informacion de ubicacion lateral
    tarjeta_ubicacion = dbc.Card([
        dbc.CardHeader(html.H4("Coordenadas y Ubicación", className="text-success m-0"), style={"fontWeight": "bold"}),
        dbc.CardBody([
            html.P([html.Strong("Continente: "), f"{ubicacion['continent']}"], className="mb-2"),
            html.P([html.Strong("Región: "), f"{ubicacion['region']}"], className="mb-2"),
            html.Hr(),
            html.P([html.Strong("Latitud: "), f"{ubicacion['latitude']:.4f}°"], className="mb-2"),
            html.P([html.Strong("Longitud: "), f"{ubicacion['longitude']:.4f}°"], className="mb-2"),
        ])
    ], style={"boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "borderRadius": "10px", "height": "100%"})

    return fig, contenedor_metricas, tarjeta_ubicacion

### Ejercicio 3.9: Ejecución del Dashboard

Con todos los componentes del layout y los callbacks definidos, procedemos a iniciar la aplicación para visualizar y probar el explorador climático interactivo.

In [87]:
# Ejecutar la aplicación final en el puerto 8053
ejecutar(app, 8053, 900)

<IPython.core.display.Javascript object>